## Test on realistic deployment input
A neutral PR-register English thoughts.
~ 30 example "user inputs" (a mix of obviously distorted thoughts, obviously neutral thoughts, and ambiguous ones) to run on the joint model, and report how it behaves.

In [1]:
# === Setup ===
from pathlib import Path

MODELS = Path("../../models")

JOINT_CHECKPOINT = MODELS / "02_04_roberta_patternreframe_none_aug" / "model"
RESULTS_DIR      = Path("./results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading from: {JOINT_CHECKPOINT}")

Loading from: ../../models/02_04_roberta_patternreframe_none_aug/model


In [2]:
# Hand-curated deployment-realistic examples.
# Each example is a single thought as would appear in the
# "automatic thoughts" field of the application after being prompted by a
# situation. They are NOT taken from any test set.

EXAMPLES = [
    # ===== Obviously distorted (one per category) =====
    # All-or-nothing
    ("distorted", "If I don't get this promotion, my whole career is a failure.",
     "All-or-nothing thinking"),
    # Catastrophizing
    ("distorted", "I made one mistake in the presentation and now everyone will think I'm incompetent.",
     "Catastrophizing"),
    # Discounting the positive
    ("distorted", "She said she liked my work but she was probably just being polite.",
     "Discounting the positive"),
    # Fortune-telling
    ("distorted", "I just know this date is going to go badly.",
     "Fortune-telling"),
    # Mind reading
    ("distorted", "He didn't text me back, so he must be angry with me.",
     "Mind reading"),
    # Labeling
    ("distorted", "I forgot to send the email. I'm such an idiot.",
     "Labeling"),
    # Mental filter
    ("distorted", "Everyone clapped except for one person who looked bored. The whole talk was a disaster.",
     "Mental filtering"),
    # Overgeneralization
    ("distorted", "I bombed this job interview. I'm never going to find work.",
     "Overgeneralization"),
    # Personalization
    ("distorted", "My partner is in a bad mood. It must be something I did.",
     "Personalization"),
    # Should statements
    ("distorted", "I should be further along in my career by now.",
     "Should statements"),

    # ===== non-distorted =====
    ("nondistorted", "I should call my mother this weekend.", "No Distortion"),
    ("nondistorted", "It looks like it might rain later.", "No Distortion"),
    ("nondistorted", "I have a meeting at 3pm tomorrow.", "No Distortion"),
    ("nondistorted", "The coffee tastes good today.", "No Distortion"),
    ("nondistorted", "I want to read more this year.", "No Distortion"),
    ("nondistorted", "My back hurts after sitting all day.", "No Distortion"),
    ("nondistorted", "I'm feeling tired and I think I'll go to bed early.", "No Distortion"),
    ("nondistorted", "The deadline is on Friday so I need to finish the report by Thursday.", "No Distortion"),
    ("nondistorted", "I disagreed with what she said but I understand her point.", "No Distortion"),
    ("nondistorted", "I made some progress today, though there's still more to do.", "No Distortion"),

    # ===== Ambiguous / borderline =====
    # Could be a fact or a distortion depending on intent
    ("ambiguous", "He didn't say hi to me at the meeting.", "uncertain"),
    # Genuine concern vs catastrophizing — depends on context
    ("ambiguous", "I'm worried I won't have enough money for retirement.", "uncertain"),
    # Realistic expectation vs all-or-nothing
    ("ambiguous", "If the funding doesn't come through, the project is over.", "uncertain"),
    # Reasonable inference vs mind reading
    ("ambiguous", "My boss has been avoiding eye contact this week.", "uncertain"),
    # Self-criticism that may be deserved
    ("ambiguous", "I shouldn't have lost my temper with him.", "uncertain"),
    # Reasonable concern vs fortune-telling
    ("ambiguous", "Given the market right now, jobs in my field are going to be hard to find.", "uncertain"),
    # Mixed positive and negative
    ("ambiguous", "I did well overall but I keep thinking about that one bad answer.", "uncertain"),
    # Reasonable doubt
    ("ambiguous", "I don't think I'm qualified for this role.", "uncertain"),
    # Self-blame that may or may not be appropriate
    ("ambiguous", "The team missed the deadline and I was the lead.", "uncertain"),
    # General malaise
    ("ambiguous", "Something doesn't feel right but I can't put my finger on what.", "uncertain"),
]

print(f"Total examples: {len(EXAMPLES)}")
print(f"  Obviously distorted:    {sum(1 for x in EXAMPLES if x[0] == 'distorted')}")
print(f"  Obviously non-distorted: {sum(1 for x in EXAMPLES if x[0] == 'nondistorted')}")
print(f"  Ambiguous / borderline:  {sum(1 for x in EXAMPLES if x[0] == 'ambiguous')}")

Total examples: 30
  Obviously distorted:    10
  Obviously non-distorted: 10
  Ambiguous / borderline:  10


In [3]:
import torch
import numpy as np
from torch.nn.functional import softmax
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(JOINT_CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(JOINT_CHECKPOINT)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()

# Label order from the checkpoint
cfg_labels = [model.config.id2label[i] for i in sorted(model.config.id2label, key=int)]
print(f"Label order from checkpoint:")
for i, lbl in enumerate(cfg_labels):
    print(f"  {i}: {lbl}")

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Label order from checkpoint:
  0: Black-and-white or polarized thinking / All or nothing thinking
  1: Catastrophizing
  2: Discounting the positive
  3: Jumping to conclusions: Fortune-telling
  4: Jumping to conclusions: mind reading
  5: Labeling and mislabeling
  6: Mental filtering
  7: No Distortion
  8: Overgeneralization
  9: Personalization
  10: Should statements


In [4]:
import pandas as pd

# Short labels for compact display
SHORT_LABEL = {
    "No Distortion":                                                    "ND",
    "Black-and-white or polarized thinking / All or nothing thinking":  "All-or-nothing",
    "Catastrophizing":                                                  "Catastrophizing",
    "Discounting the positive":                                         "Discounting +",
    "Jumping to conclusions: Fortune-telling":                          "Fortune-telling",
    "Jumping to conclusions: mind reading":                             "Mind reading",
    "Labeling and mislabeling":                                         "Labeling",
    "Mental filtering":                                                 "Mental filter",
    "Overgeneralization":                                               "Overgeneralization",
    "Personalization":                                                  "Personalization",
    "Should statements":                                                "Should",
}

# Short version of the "expected" labels (which use casual names) to
# match what the model outputs (which uses long PR names)
EXPECTED_TO_LONG = {
    "All-or-nothing thinking": "Black-and-white or polarized thinking / All or nothing thinking",
    "Catastrophizing":         "Catastrophizing",
    "Discounting the positive": "Discounting the positive",
    "Fortune-telling":         "Jumping to conclusions: Fortune-telling",
    "Mind reading":            "Jumping to conclusions: mind reading",
    "Labeling":                "Labeling and mislabeling",
    "Mental filtering":        "Mental filtering",
    "Overgeneralization":      "Overgeneralization",
    "Personalization":         "Personalization",
    "Should statements":       "Should statements",
    "No Distortion":           "No Distortion",
    "uncertain":               "uncertain",  # ambiguous examples have no canonical right answer
}

rows = []
for intent, text, expected_label in EXAMPLES:
    enc = tokenizer(text, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**enc).logits
    probs = softmax(logits, dim=-1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    pred_label = cfg_labels[pred_idx]
    max_prob = float(probs[pred_idx])

    # Second-best
    second_idx = int(np.argsort(-probs)[1])
    second_label = cfg_labels[second_idx]
    second_prob = float(probs[second_idx])

    long_expected = EXPECTED_TO_LONG[expected_label]
    if expected_label == "uncertain":
        agreement = "ambiguous"
    elif pred_label == long_expected:
        agreement = "✓ match"
    else:
        agreement = "✗ different"

    rows.append({
        "Intent":       intent,
        "Thought":      text,
        "Expected":     SHORT_LABEL.get(long_expected, expected_label),
        "Top prediction":  SHORT_LABEL[pred_label],
        "Confidence":   f"{max_prob:.2f}",
        "2nd prediction":  SHORT_LABEL[second_label],
        "2nd confidence":  f"{second_prob:.2f}",
        "Agreement":    agreement,
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))

results.to_csv(RESULTS_DIR / "deployment_examples_joint.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'deployment_examples_joint.csv'}")

      Intent                                                                                 Thought           Expected     Top prediction Confidence     2nd prediction 2nd confidence   Agreement
   distorted                            If I don't get this promotion, my whole career is a failure.     All-or-nothing     All-or-nothing       0.67    Fortune-telling           0.10     ✓ match
   distorted     I made one mistake in the presentation and now everyone will think I'm incompetent.    Catastrophizing Overgeneralization       0.26    Fortune-telling           0.24 ✗ different
   distorted                      She said she liked my work but she was probably just being polite.      Discounting +      Discounting +       0.50       Mind reading           0.23     ✓ match
   distorted                                             I just know this date is going to go badly.    Fortune-telling    Fortune-telling       0.60    Catastrophizing           0.19     ✓ match
   distorted        

In [5]:
# Summary: how often does the model match the expected label, by intent?
print("\nSummary by intent:")
for intent in ["distorted", "nondistorted", "ambiguous"]:
    sub = results[results["Intent"] == intent]
    if intent == "ambiguous":
        # For ambiguous, just show the confidence distribution
        confidences = sub["Confidence"].astype(float)
        print(f"\n{intent.upper()}: {len(sub)} examples")
        print(f"  Mean max-prob confidence: {confidences.mean():.3f}")
        print(f"  Median:                   {confidences.median():.3f}")
        print(f"  Range:                    [{confidences.min():.3f}, {confidences.max():.3f}]")
    else:
        matches = (sub["Agreement"] == "✓ match").sum()
        print(f"\n{intent.upper()}: {matches}/{len(sub)} match expected category")
        if intent == "nondistorted":
            # Which categories does the model fall back to on neutral input?
            print(f"  Predicted categories on neutral input:")
            print(sub["Top prediction"].value_counts().to_string())
            confidences = sub["Confidence"].astype(float)
            print(f"  Mean confidence on neutral input: {confidences.mean():.3f}")

# Compute a "confidence tier" breakdown for the application's UI logic
print("\nConfidence tier breakdown (for UI design):")
for tier_lo, tier_hi, name in [
    (0.0, 0.4, "low confidence (defer to user)"),
    (0.4, 0.7, "medium confidence (suggest as possibility)"),
    (0.7, 1.0, "high confidence (suggest with conviction)"),
]:
    confidences = results["Confidence"].astype(float)
    n = ((confidences >= tier_lo) & (confidences < tier_hi)).sum()
    print(f"  [{tier_lo:.1f}, {tier_hi:.1f}) {name}: {n} examples ({100*n/len(results):.0f}%)")


Summary by intent:

DISTORTED: 8/10 match expected category

NONDISTORTED: 0/10 match expected category
  Predicted categories on neutral input:
Top prediction
Fortune-telling    3
Mental filter      3
Should             2
All-or-nothing     1
Discounting +      1
  Mean confidence on neutral input: 0.457

AMBIGUOUS: 10 examples
  Mean max-prob confidence: 0.501
  Median:                   0.545
  Range:                    [0.210, 0.850]

Confidence tier breakdown (for UI design):
  [0.0, 0.4) low confidence (defer to user): 8 examples (27%)
  [0.4, 0.7) medium confidence (suggest as possibility): 18 examples (60%)
  [0.7, 1.0) high confidence (suggest with conviction): 4 examples (13%)
